# Building `build_sample` step by step

Run each cell and look at the actual DataFrame output before moving to the next one.

In [12]:
import sys
sys.path.insert(0, "../src")

import random
from pathlib import Path

import pandas as pd
from config import KT1_RAW_DIR, KT1_STREAM_COLUMNS

pd.set_option("display.max_columns", None)

In [13]:
def list_user_files(kt1_dir: Path) -> list[Path]:
    files = sorted(kt1_dir.glob("u*.csv"))
    if not files:
        raise FileNotFoundError(f"No user files found under {kt1_dir}")
    return files


def sample_user_files(files: list[Path], n: int, seed: int) -> list[Path]:
    random.seed(seed)
    k = min(n, len(files))
    return random.sample(files, k)


DTYPES = {
    "timestamp": "int64",
    "solving_id": "int64",
    "question_id": "string",
    "user_answer": "string",
    "elapsed_time": "int64",
}


def load_and_tag(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=DTYPES)
    df["user_id"] = path.stem
    return df[KT1_STREAM_COLUMNS]

In [14]:
files = list_user_files(KT1_RAW_DIR)
print(len(files), "files found")
files[:5]

784309 files found


[PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u1.csv'),
 PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u10.csv'),
 PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u100.csv'),
 PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u1000.csv'),
 PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u10000.csv')]

In [15]:
# small n so it's fast to inspect visually
chosen = sample_user_files(files, n=5, seed=42)
chosen

[PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u721760.csv'),
 PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u205661.csv'),
 PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u123721.csv'),
 PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u93899.csv'),
 PosixPath('/home/verseua/projects/PySpark-Data-Processing/data/raw/kt1/KT1/u362877.csv')]

In [16]:
one_df = load_and_tag(chosen[0])
print(one_df.shape)
one_df.head()

(27, 6)


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
0,1565576049229,1,q8700,a,10000,u721760
1,1565576077454,2,q146,b,23000,u721760
2,1565576091480,3,q5740,b,12000,u721760
3,1565576103559,4,q6731,b,10000,u721760
4,1565576116667,5,q3974,b,11000,u721760


## Step 1 — load and tag every chosen user

Fill in the blank: call `load_and_tag` on every path in `chosen`.

In [17]:
frames = [load_and_tag(p) for p in chosen]

print(len(frames), "frames")
for f in frames:
    print(f.shape, f["user_id"].iloc[0])

5 frames
(27, 6) u721760
(67, 6) u205661
(4, 6) u123721
(30, 6) u93899
(10, 6) u362877


## Step 2 — stack them into one DataFrame

Fill in the blank: which pandas function stacks a list of DataFrames on top of each other (rows), as opposed to joining them side by side on a key?

In [18]:
merged = pd.concat(frames, ignore_index=True)
print(merged.shape)
merged["user_id"].value_counts()  # notice rows from all 5 users are now in one table

(138, 6)


user_id
u205661    67
u93899     30
u721760    27
u362877    10
u123721     4
Name: count, dtype: int64

## Step 3 — sort by timestamp across all users

Look at `merged.head(10)` before running the next cell — the rows are grouped by user, not interleaved by time yet. This step is what actually creates the shared timeline.

Fill in the two blanks: which column do we sort by, and which `kind` keeps tied timestamps in a consistent, repeatable order?

In [19]:
merged.sort_values("timestamp", inplace=True, kind="stable")
merged.head(10)  # now watch user_id jump around — that's the interleaved timeline

,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
98,1515520599729,1,q8098,b,18000,u93899
99,1515520622662,2,q8074,b,20000,u93899
100,1515520645710,3,q176,c,20000,u93899
101,1515520665015,4,q1279,b,17000,u93899
102,1515520739210,5,q2067,a,24000,u93899
103,1515520739226,5,q2068,a,24000,u93899
104,1515520739312,5,q2069,d,24000,u93899
105,1515520800779,6,q3413,d,19666,u93899
106,1515520800875,6,q3412,d,19666,u93899
107,1515520800886,6,q3411,d,19666,u93899


## Step 4 — clean up the index

Check `merged.index` now — sorting scrambled it (it's not 0,1,2,... in order anymore). Fill in the blank: should `drop` be `True` or `False`? Think about whether you want the old scrambled index thrown away, or kept around as a new column.

In [20]:
merged.reset_index(drop=False, inplace=True)

merged.head(10)

,index,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
0,98,1515520599729,1,q8098,b,18000,u93899
1,99,1515520622662,2,q8074,b,20000,u93899
2,100,1515520645710,3,q176,c,20000,u93899
3,101,1515520665015,4,q1279,b,17000,u93899
4,102,1515520739210,5,q2067,a,24000,u93899
5,103,1515520739226,5,q2068,a,24000,u93899
6,104,1515520739312,5,q2069,d,24000,u93899
7,105,1515520800779,6,q3413,d,19666,u93899
8,106,1515520800875,6,q3412,d,19666,u93899
9,107,1515520800886,6,q3411,d,19666,u93899


## Now turn Steps 1–4 into a real function

Same logic as above, just packaged so it can be called once with the full user list instead of typed out step by step.

In [21]:
def build_sample(files: list[Path]) -> pd.DataFrame:
    frames = [load_and_tag(p) for p in files]
    merged = pd.concat(frames, ignore_index=True)
    merged.sort_values("timestamp", inplace=True, kind="stable")
    merged.reset_index(drop=True, inplace=True)
    return merged


# quick sanity check against the same 5-user sample from before
test_sample = build_sample(chosen)
print(test_sample.shape)
test_sample.head(10)

(138, 6)


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
0,1515520599729,1,q8098,b,18000,u93899
1,1515520622662,2,q8074,b,20000,u93899
2,1515520645710,3,q176,c,20000,u93899
3,1515520665015,4,q1279,b,17000,u93899
4,1515520739210,5,q2067,a,24000,u93899
5,1515520739226,5,q2068,a,24000,u93899
6,1515520739312,5,q2069,d,24000,u93899
7,1515520800779,6,q3413,d,19666,u93899
8,1515520800875,6,q3412,d,19666,u93899
9,1515520800886,6,q3411,d,19666,u93899


## Run it for real

This is the one cell that actually matters for the pipeline: pick 500 users, build the merged/sorted sample, write it to `data/sample/kt1_sample.csv`. `feeder_dev.ipynb` reads this file.

In [22]:
from config import SAMPLE_FILE, DEFAULT_N_USERS, RANDOM_SEED

all_files = list_user_files(KT1_RAW_DIR)
real_chosen = sample_user_files(all_files, DEFAULT_N_USERS, RANDOM_SEED)
real_sample = build_sample(real_chosen)

SAMPLE_FILE.parent.mkdir(parents=True, exist_ok=True)
real_sample.to_csv(SAMPLE_FILE, index=False)

print(f"Wrote {len(real_sample)} rows from {len(real_chosen)} users to {SAMPLE_FILE}")
real_sample.head()

Wrote 43574 rows from 500 users to /home/verseua/projects/PySpark-Data-Processing/data/sample/kt1_sample.csv


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
0,1498906740412,1,q129,b,27000,u22094
1,1498906764080,2,q8058,b,22000,u22094
2,1498906786376,3,q8120,a,20000,u22094
3,1498906818824,4,q157,d,30000,u22094
4,1498906848124,5,q52,b,27000,u22094
